# Playground for functions and tests on the cloud point files

## Libs

In [11]:
import open3d as o3d
import os
import numpy as np
import matplotlib.pyplot as plt
import random

## Import Files

In [8]:
filepath = os.path.join("data", "pdc", "pdc_0001.ply") 


pcd_paths = [
    os.path.join(r"Test_part", "Verification_examples", "01_Bottom_CubeSat_MODEL.pcd"),
    os.path.join(r"Test_part", "Verification_examples", "02_Wall_no_Logo_1.pcd"),
    os.path.join(r"Test_part", "Verification_examples", "02_Wall_with_acor_logo.pcd"),
    os.path.join(
        r"Test_part",
        "Verification_examples",
        "02_wall_with_PLCM_logo_plcm_02_Wall_no_Logo.pcd",
    ),
    os.path.join(
        r"Test_part",
        "Verification_examples",
        "02_wall_with_PLCM_logo_plcm_PLCM_Logo.pcd",
    ),
    os.path.join(r"Test_part", "Verification_examples", "03_Board_Empty.pcd"),
]
# pringles_path = os.path.join(r"Test_part", "merged_cloud.ply")
pringles_path = os.path.join(r"Test_part", "pringles", "tsdf", "nontextured.ply")
def load_point_cloud(file_path):
    pcd = o3d.io.read_point_cloud(file_path)
    pcd.remove_non_finite_points()
    return pcd
# pcd = load_point_cloud(pcd_paths[0])
pcd = load_point_cloud(pringles_path)

In [4]:
o3d.visualization.draw_geometries([pcd])

### Downsample and estimate normals

In [9]:
pcd_down = pcd.voxel_down_sample(0.003)
o3d.visualization.draw_geometries([pcd_down])

In [10]:
pcd_down.estimate_normals()
# pcd_down.normals = np.array(pcd_down.normalize_normals())

In [8]:
import random

def generate_grasp_candidates_sphere(pcd, num_candidates=100, visualize=True):
    """
    Gera candidatos a pega lançando raios de uma esfera envolvente em direção ao centro.
    Retorna:
        - points: Lista de pontos de superfície encontrados (candidatos).
        - normals: Lista de normais nesses pontos (vetor de aproximação inverso).
    """
    # 1. Calcular o Centro e o Raio da Esfera Envolvente
    min_bound = pcd.get_min_bound()
    max_bound = pcd.get_max_bound()
    center = (min_bound + max_bound) / 2
    
    # O raio deve ser grande o suficiente para cobrir todo o objeto
    # Usamos a maior dimensão / 2 e adicionamos uma margem de segurança (ex: 10%)
    max_dim = np.max(max_bound - min_bound)
    radius = (max_dim / 2) * 1.5 

    candidates = []
    candidate_normals = []
    
    # Precisamos de um modelo RayCasting para o Open3D funcionar
    # O RayCasting requer uma MALHA (Mesh), não uma nuvem de pontos.
    # Por isso, vamos criar uma aproximação convexa rápida (Convex Hull) apenas para os raios baterem.
    # (Nota: O Justus sugeriu STL, o Convex Hull é a versão rápida disso gerada na hora)
    mesh, _ = pcd.compute_convex_hull()
    ray_scene = o3d.t.geometry.RaycastingScene()
    
    # Converter a malha legacy para tensor (necessário para versões novas do Open3D)
    mesh_t = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
    ray_scene.add_triangles(mesh_t)

    print(f"Gerando {num_candidates} candidatos...")

    for _ in range(num_candidates):
        # 2. Gerar um ponto aleatório na superfície da esfera virtual
        # Usamos coordenadas esféricas
        theta = random.uniform(0, 2 * np.pi)
        phi = random.uniform(0, np.pi)
        
        x = center[0] + radius * np.sin(phi) * np.cos(theta)
        y = center[1] + radius * np.sin(phi) * np.sin(theta)
        z = center[2] + radius * np.cos(phi)
        
        sphere_point = np.array([x, y, z], dtype=np.float32)
        
        # 3. Criar o Raio: Da Esfera -> Para o Centro
        direction = center - sphere_point
        direction = direction / np.linalg.norm(direction) # Normalizar
        
        # Formato do raio para o Open3D: [origem_x, origem_y, origem_z, dir_x, dir_y, dir_z]
        ray = np.concatenate([sphere_point, direction]).astype(np.float32)
        
        # 4. Disparar o Raio
        # cast_rays espera uma lista de raios, passamos um.
        result = ray_scene.cast_rays(o3d.core.Tensor([ray]))
        
        # Verificar se bateu (t_hit é a distância. Se for inf, não bateu)
        t_hit = result['t_hit'].numpy()[0]
        
        if t_hit < float('inf'):
            # Calcular o ponto exato de colisão no objeto
            hit_point = sphere_point + t_hit * direction
            
            # A "Normal" de aproximação é oposta à direção do raio
            # (Ou podemos usar a normal da superfície se o raycast devolver, o 'primitive_normals')
            hit_normal = result['primitive_normals'].numpy()[0]
            
            candidates.append(hit_point)
            candidate_normals.append(hit_normal)

    # Visualização (Opcional)
    if visualize:
        print(f"Visualizando {len(candidates)} candidatos encontrados (Vermelho)...")
        pcd.paint_uniform_color([0.8, 0.8, 0.8]) # Objeto cinza
        
        cand_pcd = o3d.geometry.PointCloud()
        cand_pcd.points = o3d.utility.Vector3dVector(np.array(candidates))
        cand_pcd.paint_uniform_color([1, 0, 0]) # Candidatos vermelhos
        
        # Desenhar a esfera envolvente (arame) para referência
        # sphere_frame = o3d.geometry.TriangleMesh.create_sphere(radius).get_wireframe_geometry()
        # CÓDIGO CORRETO
        sphere_mesh = o3d.geometry.TriangleMesh.create_sphere(radius)
        sphere_frame = o3d.geometry.LineSet.create_from_triangle_mesh(sphere_mesh)
        sphere_frame.translate(center)
        sphere_frame.paint_uniform_color([0, 0, 1])

        o3d.visualization.draw_geometries([pcd, cand_pcd, sphere_frame])

    return np.array(candidates), np.array(candidate_normals)

# --- COMO TESTAR ---
# pcd = o3d.io.read_point_cloud("caminho/para/seu/arquivo.ply")
# pcd.estimate_normals() # Garante que tem normais para o convex hull
# points, normals = generate_grasp_candidates_sphere(pcd)

In [39]:
min_bound = pcd.get_min_bound()
max_bound = pcd.get_max_bound()
tamanho = max_bound - min_bound

print(f"Tamanho do objeto (X, Y, Z): {tamanho}")

Tamanho do objeto (X, Y, Z): [0.08677229 0.0856603  0.24439824]


In [11]:

# --- COMO TESTAR ---
points, normals = generate_grasp_candidates_sphere(pcd_down, 1000)

Gerando 1000 candidatos...
Visualizando 1000 candidatos encontrados (Vermelho)...


In [14]:
pcd_down.get_max_bound() - pcd_down.get_min_bound()

array([0.08342795, 0.08610535, 0.24055953])